In [1]:
from pathlib import Path 

list(Path(".").glob("*.csv"))

[WindowsPath('WA_Fn-UseC_-HR-Employee-Attrition.csv')]

In [2]:
import pandas as pd

file_path = Path("WA_Fn-UseC_-HR-Employee-Attrition.csv")
hr = pd.read_csv(file_path)

print("Rows and columns:", hr.shape)
hr.head()


Rows and columns: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
print("Missing cells:", hr.isna().sum().sum())
print("Unique employee numbers:", hr["EmployeeNumber"].nunique())
print("Duplicate employee numbers:", hr["EmployeeNumber"].duplicated().sum())


Missing cells: 0
Unique employee numbers: 1470
Duplicate employee numbers: 0


In [6]:
overtime_summary = (
    hr.groupby("OverTime")["Attrition"]
      .agg(
          employees="size",
          left=lambda values: (values == "Yes").sum()
      )
)

overtime_summary["attrition_rate in %"] = ((
    overtime_summary["left"] / overtime_summary["employees"]
)*100)

overtime_summary

,employees,left,attrition_rate in %
OverTime,,,
No,1054,110,10.436433
Yes,416,127,30.528846


In [7]:
hr_analysis = hr.copy()

hr_analysis["AgeBand"] = pd.cut(
    hr_analysis["Age"],
    bins=[0, 25, 35, 45, 55, float("inf")],
    labels=["Under 25", "25–34", "35–44", "45–54", "55+"],
    right=False
)

hr_analysis["TenureBand"] = pd.cut(
    hr_analysis["YearsAtCompany"],
    bins=[0, 3, 6, 11, float("inf")],
    labels=["0–2 years", "3–5 years", "6–10 years", "11+ years"],
    right=False
)

hr_analysis["IncomeBand"] = pd.cut(
    hr_analysis["MonthlyIncome"],
    bins=[0, 3000, 5000, 8000, 12000, float("inf")],
    labels=["Under 3,000", "3,000–4,999", "5,000–7,999", "8,000–11,999", "12,000+"],
    right=False
)

hr_analysis.shape

(1470, 38)

In [8]:
processed_dir = Path("data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "hr_attrition_analysis_ready.csv"
hr_analysis.to_csv(output_path, index=False)

print(f"Saved: {output_path}")

Saved: data\processed\hr_attrition_analysis_ready.csv
